# IFB 01 — Templates & Incremental Loader

Create the operational CSV templates and inspect any incremental data files already available.

In [1]:
# Import libraries
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# Define config paths
PROJECT_ROOT = Path.cwd()
while (
    not (PROJECT_ROOT / "src").exists()
    and PROJECT_ROOT != PROJECT_ROOT.parent
):
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = (
    PROJECT_ROOT
    / "configs"
    / "inference_feature_builder.yaml"
)

print("Project root:", PROJECT_ROOT)
print("IFB config:", CONFIG_PATH)


Project root: e:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk
IFB config: e:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\configs\inference_feature_builder.yaml


In [3]:
# Import module for running inference_feature_builder
from src.ontario_peak_risk.inference_feature_builder.io import (
    load_ifb_config,
    load_operational_updates,
    load_project_history,
    resolve_path,
)
from src.ontario_peak_risk.inference_feature_builder.templates import (
    generate_templates,
)


In [4]:
# Define paths
config, project_root = load_ifb_config(CONFIG_PATH)

EXAMPLE_FORECAST_ORIGIN = pd.Timestamp("2026-08-27 00:00:00")

paths = generate_templates(
    resolve_path(project_root, config["paths"]["templates_dir"]),
    forecast_origin=EXAMPLE_FORECAST_ORIGIN,
    fsas=config["inference"]["fsas"],
)
paths


{'demand': WindowsPath('E:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/DataLocal/ontario-electricity-peak-risk/data/operational/templates/demand_update_template.csv'),
 'weather_history': WindowsPath('E:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/DataLocal/ontario-electricity-peak-risk/data/operational/templates/weather_history_update_template.csv'),
 'weather_forecast': WindowsPath('E:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/DataLocal/ontario-electricity-peak-risk/data/operational/templates/weather_forecast_template.csv')}

In [5]:
# Load available data
project_history = load_project_history(config, project_root)
updates = load_operational_updates(config, project_root)

In [7]:
# View project history data
print(
    "Project history:",
    project_history["timestamp"].min(),
    "to",
    project_history["timestamp"].max(),
)

Project history: 2021-01-01 00:00:00 to 2025-12-31 23:00:00


In [6]:
# View available Operational data 
print("\nOperational files discovered:")
display(updates["inventory"])


Operational files discovered:


,source_type,file,path,rows,min_timestamp,max_timestamp
0,demand_update,hourly_consumption_fsa_L4T_20260101-20260430.csv,E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Cap...,2880,2026-01-01,2026-04-30 23:00:00
1,demand_update,hourly_consumption_fsa_M5R_20260101-20260430.csv,E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Cap...,2880,2026-01-01,2026-04-30 23:00:00
2,demand_update,hourly_consumption_fsa_M5S_20260101-20260430.csv,E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Cap...,2880,2026-01-01,2026-04-30 23:00:00
3,demand_update,hourly_consumption_fsa_M6G_20260101-20260430.csv,E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Cap...,2880,2026-01-01,2026-04-30 23:00:00
4,demand_update,hourly_consumption_fsa_M9R_20260101-20260430.csv,E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Cap...,2880,2026-01-01,2026-04-30 23:00:00
5,demand_update,hourly_consumption_fsa_M9W_20260101-20260430.csv,E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Cap...,2880,2026-01-01,2026-04-30 23:00:00
6,weather_forecast,weather_forecast_L4T_20250701-20260728.csv,E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Cap...,9432,2025-07-01,2026-07-28 23:00:00
7,weather_forecast,weather_forecast_M5R_20250701-20260728.csv,E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Cap...,9432,2025-07-01,2026-07-28 23:00:00
8,weather_forecast,weather_forecast_M5S_20250701-20260728.csv,E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Cap...,9432,2025-07-01,2026-07-28 23:00:00
9,weather_forecast,weather_forecast_M6G_20250701-20260728.csv,E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Cap...,9432,2025-07-01,2026-07-28 23:00:00


### Note:
- Add new CSV files over time; the loader combines them automatically.
- You do not need to manually recreate one large historical file.